# P2H2P — Study 2: what matters **below ground**

Nine things you can choose about the well and what goes in it, and one
question: which of them changes the answer, and which pairs of them only
matter together?

*Runner v1 · built 2026-09-30 17:25 BRT · model version is fetched and printed below*

---

### The idea behind splitting the problem in two

The plant is a chain, and only one thing passes between the machines at the
surface and the rock below: **hot water in a pipe**. Five numbers describe
that water completely —

| | |
|---|---|
| how hot it goes down when charging | $T_{4c}$ |
| how much cooler it comes back | the charge glide |
| how cold it goes down when discharging | $T_{3d}$ |
| how much warmer it comes back | the discharge glide |
| how much heat has to move each cycle | the duty |

Hold those five fixed and the two halves stop talking to each other. The heat
pump does not care what the storage material is made of; the rock does not
care what machine is pushing the water. So this notebook **freezes all five**
and varies only what is underground.

That is why the earlier thirteen-factor screen was hard to read: six of its
factors were surface factors and three were underground ones, and they were
all in the same ranking, competing for the same attention.

### Why a factorial here, and not another Morris screen

Morris was chosen when we expected the model to be badly non-linear. It
mostly is not — in the v0.16 screen, eleven of thirteen factors pushed the
same direction every single time they were nudged. Morris's advantage never
appeared, but its cost did: **it cannot tell you about pairs.**

And pairs are the interesting part down here. *Are fins worth adding only
when the material conducts badly? Does a bigger pipe help only in a deep
well?* A factorial answers both, in the same run, for free.

### What it costs

128 corner runs + 4 centre runs, about 38 s each → **roughly 85 minutes.**
The cell checkpoints every 16 runs, so a Colab disconnect costs you a few
minutes, not the whole study.

---
## 1. Setup

`CoolProp` is not in a stock Colab image. The model and both runners are
pulled from the repository — `doe_borehole` borrows the progress bar from
`doe_morris`, so both come down.

In [ ]:
!pip install -q CoolProp

In [ ]:
import urllib.request, pathlib, sys, re

BRANCH = 'notebook-model'          # pin a tag here for a quotable run
RAW = ('https://raw.githubusercontent.com/JaderBarbosaLSU/'
       'wellbore-thermal-storage/' + BRANCH)

for name in ('thums.py', 'doe_morris.py', 'doe_borehole.py'):
    urllib.request.urlretrieve(RAW + '/model_src/' + name, name)
    print(f'{name:>16}  {len(open(name).read().splitlines()):>5} lines')

# DROP THE CACHED MODULES BEFORE IMPORTING.
# `urlretrieve` overwrites the file on disk, but `import` returns whatever is
# already in sys.modules -- so re-running this cell after the repository has
# moved silently keeps the OLD code, and the next cell dies with an
# AttributeError on something that plainly exists in the file you just
# downloaded. That exact failure cost an hour on the Morris notebook. Pop in
# dependency order: doe_borehole imports doe_morris imports thums.
for _m in ('doe_borehole', 'doe_morris', 'thums'):
    sys.modules.pop(_m, None)

import thums as T, doe_borehole as db
MODEL_VERSION = re.search(r'live model, v([\d.]+?)\.\s',
                          open('thums.py').read()).group(1)
print(f'\nmodel v{MODEL_VERSION} · {len(db.FACTORS)} factors · '
      f'{len(db.KPIS)} KPIs')
assert db.Progress is not None, (
    'stale modules in the kernel -- Runtime > Restart session')

---
## 2. What is being varied

**Edit `db.FACTORS` here, not in the file.** Each range is argued in a comment
beside it in `doe_borehole.py`.

| | factor | low | high | what it is |
|---|---|---|---|---|
| A | `h_m_kJkg` | 180 | 380 kJ/kg | **latent heat** — how much energy a kilogram of the material absorbs while melting. The single biggest lever material choice gives you. |
| B | `k_scale` | 0.5× | 2.0× | **thermal conductivity** — how fast heat gets into the material. Scales solid and liquid together so the ratio between them stays physical. |
| C | `rho_scale` | 0.85× | 1.15× | **density** — how many kilograms fit in the hole. |
| D | `cp_scale` | 0.8× | 1.2× | **specific heat** — the *sensible* storage either side of melting. |
| E | `L_ft` | 3000 | 8000 ft | **well depth**. Deeper wells hold more but cost more to pump through. |
| F | `num_tubes` | 1 | 3 | **hairpins per hole** — how many U-tubes you hang down one well. |
| G | `nps` | 1¼″ | 2″ | **pipe size**, Sch 80. Bigger pipe pumps easier but crowds the hole. |
| H | `num_fins` | 8 | 32 | **how many fins** on each tube. |
| J | `N_lay` | 5 | 15 | **cascade layers** — how finely the melting temperature is stepped down the well. |

### Three things deliberately held fixed

**Fin length, at 3.5 mm.** Only the fin *count* varies. This makes the
question *"how many fins?"* — which is what a manufacturer quotes — rather
than the muddier *"how much fin?"*. 3.5 mm also fits inside the tightest
geometry in the design (three hairpins of 2″ pipe), so no corner of the
design is thrown away for not being a shape.

**Fouling.** It came last of thirteen in the v0.16 screen. Sampling it would
spend one of only nine columns on a known non-effect.

**The temperature span of the cascade,** at 48.9 K. This one matters and is
easy to miss: if `N_lay` were allowed to change the span, then changing the
number of layers would also move the temperature the ORC sees — a *surface*
effect leaking into a borehole study. Pinned this way, `N_lay` varies only the
**granularity** of the cascade, which is the thing we actually want to ask
about.

In [ ]:
import pandas as pd
rows = []
for name, (lo, hi, kind) in db.FACTORS.items():
    if kind == 'nps':
        lo, hi = f'{db.NPS_LIST[0]}"', f'{db.NPS_LIST[1]}"'
    rows.append({'letter': db.LETTERS[len(rows)], 'factor': name,
                 'low': lo, 'high': hi})
display(pd.DataFrame(rows).set_index('letter'))
print('frozen (the interface + the three above):')
for k, v in db.FROZEN.items():
    print(f'   {k:>16} = {v}')
print(f'   {"cascade span":>16} = {db.SPAN_FIXED:.2f} K')

---
## 3. The design, checked rather than trusted

Nine factors at two levels each is $2^9 = 512$ combinations. We run 128 of
them — one in four — chosen so that nothing we care about gets tangled with
anything else we care about.

The risk with running a subset is **confounding**: two different effects
landing in the same column, so the run cannot tell them apart and you get one
number that is secretly the sum of two. The design below is built so that all
9 single-factor effects and all 36 pairs are mutually clean.

The cell proves it rather than asserting it. It builds all 45 columns and
checks that every pair of them is exactly perpendicular. If that check ever
fails, the interaction table later in this notebook is fiction — so it runs
*before* the 85 minutes, not after.

> This design is **Resolution VI**, which is one rung better than the
> eleven-factor version could manage. Dropping two factors bought that.
> In plain terms: for a pair effect to be corrupted here, a *four-way*
> conspiracy between factors would have to be real and large.

In [ ]:
assert db.verify_design(), 'design is confounded -- do not run'

### Does every corner exist as a real object?

A design point is a set of numbers; it is not automatically a *thing you can
build*. Three hairpins of 2″ pipe with long fins do not fit in a 7″ hole.

That matters more than it sounds. When corners drop out they rarely drop out
at *random* — the failures share a cause, so the loss lands on specific
columns and quietly corrupts exactly the interactions those columns measure.
An earlier version of this study lost 16 of 128 corners that way, all of them
the same shape. Fixing the fin length at 3.5 mm removes the problem at the
source.

In [ ]:
import numpy as np
X = db.design_matrix()
clear = [db.build_case(r).r_cell - db.build_case(r).r_e - db.build_case(r).fin_L
         for r in X]
bad = sum(1 for r in X if not db.geometry_ok(db.build_case(r)))
print(f'corners that are not a geometry: {bad} of {len(X)}')
print(f'tightest clearance anywhere:     {min(clear)*1000:.2f} mm')

---
## 4. Run it

**~85 minutes.** The bar shows elapsed time and an estimate of what is left,
so you can tell a slow run from a stuck one.

It writes `borehole_raw.json` every 16 runs. If Colab drops you, re-run the
cell — you lose at most a few minutes.

The four extra runs at the end are all at the **centre** of every range, and
they are the insurance on the whole method. A two-level design assumes the
response runs in a straight line between the corners it visited. Running the
middle a few times and comparing it against what the corners *predict* for the
middle turns that assumption into a measurement. If the two disagree, the
effects below are still directionally right but should not be read as
magnitudes. Four runs, three minutes.

In [ ]:
rows = db.run(centre_points=4, out='borehole_raw.json')

In [ ]:
from google.colab import files
files.download('borehole_raw.json')

---
## 5. What moved

An **effect** here is the plainest thing it could be: how much the answer
changes when you take a factor from its low value to its high value, averaged
over every combination of everything else. Not a fitted slope — a difference
of averages.

Because the design is balanced, "averaged over everything else" is honest:
each factor is at its high level in exactly half the runs, and in those two
halves every *other* factor is still equally represented.

In [ ]:
res, raw = db.analyse('borehole_raw.json')
n_ok = sum(1 for r in raw['rows'] if r['kpi'])
print(f'{n_ok} of {len(raw["rows"])} runs usable')
display(db.top(res, 'psi', n=9, kind='main'))

### Main effects, side by side

Bars are in **% of the mean** so the KPIs can share an axis. Sign is
direction: a positive bar means raising that factor raises that KPI.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SHOW = ['psi', 'eta_RTE', 'wells_per_MWe', 'rho_E_kWh_m3']
names = list(db.FACTORS)
fig, axes = plt.subplots(1, len(SHOW), figsize=(4.0 * len(SHOW), 4.6),
                         sharey=True)
for ax, kpi in zip(axes, SHOW):
    e = res[kpi]['effects']
    v = np.array([100 * e[n] / res[kpi]['mean'] for n in names])
    o = np.argsort(np.abs(v))
    ax.barh(range(len(v)), v[o],
            color=['#c0392b' if x < 0 else '#2471a3' for x in v[o]])
    ax.set_yticks(range(len(v)))
    ax.set_yticklabels([names[i] for i in o], fontsize=9)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(kpi, fontsize=11)
    ax.set_xlabel('effect, % of mean')
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Study 2 — single-factor effects below ground', fontsize=13)
fig.tight_layout()
fig.savefig('borehole_main.pdf')
plt.show()

---
## 6. What only matters in company

This is the part Morris could not give us.

A **pair effect** answers: *does the value of this factor depend on where that
one is set?* A large pair effect means the two cannot be chosen separately —
deciding about fins without first deciding about conductivity would be a
mistake. A small one means they are independent, and you may optimise them one
at a time, which is far cheaper.

Read the table as: `X x Y` is *half the difference* between X's effect when Y
is high and X's effect when Y is low.

In [ ]:
display(db.top(res, 'psi', n=10, kind='inter'))

### The heat map

Everything at once. Dark cells are pairs that must be decided together;
near-white cells are pairs you may treat as independent. The diagonal is the
single-factor effect, for scale — a pair only deserves attention if it is a
noticeable fraction of the diagonal it sits between.

In [ ]:
KPI = 'psi'          # change and re-run for any KPI in db.KPIS
e = res[KPI]['effects']
m = res[KPI]['mean']
k = len(names)
M = np.full((k, k), np.nan)
for i in range(k):
    M[i, i] = 100 * e[names[i]] / m
    for j in range(i + 1, k):
        val = 100 * e[f'{names[i]} x {names[j]}'] / m
        M[i, j] = M[j, i] = val

fig, ax = plt.subplots(figsize=(7.6, 6.6))
lim = np.nanmax(np.abs(M))
im = ax.imshow(M, cmap='RdBu_r', vmin=-lim, vmax=lim)
ax.set_xticks(range(k)); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(k)); ax.set_yticklabels(names)
for i in range(k):
    for j in range(k):
        ax.text(j, i, f'{M[i, j]:.1f}', ha='center', va='center', fontsize=7,
                color='white' if abs(M[i, j]) > 0.55 * lim else 'black')
ax.set_title(f'{KPI}: diagonal = single factor, off-diagonal = pair\n'
             '(% of mean)', fontsize=11)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig('borehole_interactions.pdf')
plt.show()

### Looking at one pair directly

Pick the strongest pair and draw it the old-fashioned way: the KPI against
factor 1, once with factor 2 low and once with it high. **Parallel lines mean
no interaction** — factor 2 shifts the answer but does not change what factor
1 is worth. Lines that converge, cross or fan out mean the two decisions are
entangled.

In [ ]:
import itertools
pairs = {kk: vv for kk, vv in res[KPI]['effects'].items() if ' x ' in kk}
best = max(pairs, key=lambda kk: abs(pairs[kk]))
fa, fb = best.split(' x ')
ia, ib = names.index(fa), names.index(fb)

corner = [r for r in raw['rows'] if not r['centre'] and r['kpi']]
Xc = np.array([r['x'] for r in corner])
y = np.array([r['kpi'][KPI] for r in corner])

fig, ax = plt.subplots(figsize=(5.6, 4.4))
for sb, style, lab in [(-1, 'o-', 'low'), (+1, 's--', 'high')]:
    ys = [y[(Xc[:, ib] == sb) & (Xc[:, ia] == sa)].mean() for sa in (-1, +1)]
    ax.plot([0, 1], ys, style, lw=2, ms=8, label=f'{fb} {lab}')
ax.set_xticks([0, 1]); ax.set_xticklabels([f'{fa}\nlow', f'{fa}\nhigh'])
ax.set_ylabel(KPI); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f'strongest pair: {best}', fontsize=11)
fig.tight_layout(); fig.savefig('borehole_pair.pdf'); plt.show()
print(f'pair effect = {pairs[best]:.5f} '
      f'({100 * pairs[best] / res[KPI]["mean"]:.1f} % of mean)')

---
## 7. Is the straight line a fair description?

The centre points, cashed in. `curv_rel` is the gap between the measured
middle and the predicted middle, in units of the spread across the corners.

| `curv_rel` | reading |
|---|---|
| under ~0.2 | the plane is a fair description; quote the effects as magnitudes |
| 0.2 – 0.5 | directions are sound, magnitudes are approximate |
| over ~0.5 | there is real curvature; this KPI needs a three-level study before anyone quotes a number |

A KPI failing this test is **not** a failed run. It is the design telling you
what it is not entitled to say, which is the thing a two-level design is worst
at noticing on its own.

In [ ]:
import pandas as pd
cur = pd.DataFrame({
    kpi: {'mean': res[kpi]['mean'],
          'curvature': res[kpi]['curvature'],
          'curv_rel': res[kpi]['curv_rel']}
    for kpi in res}).T.sort_values('curv_rel', key=np.abs, ascending=False)
display(cur.round(4))

---
## 8. The one-page answer

Every factor, every KPI, ranked. This is the table the paper wants: a reader
should be able to look down a column and see what to spend money on.

In [ ]:
S = pd.DataFrame({
    kpi: {n: 100 * res[kpi]['effects'][n] / res[kpi]['mean'] for n in names}
    for kpi in SHOW})
S['|mean| across KPIs'] = S.abs().mean(axis=1)
display(S.sort_values('|mean| across KPIs', ascending=False).round(1))
S.to_csv('borehole_summary.csv')
from google.colab import files
files.download('borehole_summary.csv')

---
### What this study can and cannot tell you

**Can:** which underground choices move the plant, by how much, in which
direction, and which of them have to be decided together.

**Cannot:**

- *Anything about the surface machines.* They were frozen on purpose. That is
  Study 1.
- *Anything about the interface itself* — the four water temperatures and the
  duty. Those are frozen too, and they are where the two halves meet, so they
  need their own study.
- *An optimum.* Corners and a midpoint locate a direction, not a peak. If
  section 7 shows curvature, a three-level design over the two or three
  factors that survived is the natural next step — and it will be cheap,
  because most of the nine will not survive.